# Retrieve and save job results

In [1]:
import datetime
from qiskit_ibm_runtime import QiskitRuntimeService

three_months_ago = datetime.datetime.now() - datetime.timedelta(days=90)

service = QiskitRuntimeService()
jobs_in_last_three_months = service.jobs(created_after=three_months_ago)
jobs_in_last_three_months[:3]  # show first three jobs

[<RuntimeJobV2('d7h30hrjne2c7393jqpg', 'sampler')>,
 <RuntimeJobV2('d7h30bq2khts739p7qn0', 'sampler')>,
 <RuntimeJobV2('d7h2u73jne2c7393joeg', 'estimator')>]

In [2]:
# Get ID of most recent successful job for demonstration.
# This will not work if you've never successfully run a job.
successful_job = next(
    j for j in service.jobs(limit=1000) if j.status() == "DONE"
)
job_id = successful_job.job_id()
print(job_id)

qiskit_runtime_service._create_backend_obj:WARNING:2026-05-05 23:26:38,191: Unable to create configuration for ibm_torino. '404 Client Error: Not Found for url: https://quantum.cloud.ibm.com/api/v1/backends/ibm_torino/configuration. {"errors":[{"code":"not_found","message":"device not found","more_info":"https://cloud.ibm.com/apidocs/quantum-computing#error-handling"}],"trace":"786cf463-af6d-4c2a-a8db-50fe153214f9"}\n' 
qiskit_runtime_service._create_backend_obj:WARNING:2026-05-05 23:26:38,747: Unable to create configuration for ibm_torino. '404 Client Error: Not Found for url: https://quantum.cloud.ibm.com/api/v1/backends/ibm_torino/configuration. {"errors":[{"code":"not_found","message":"device not found","more_info":"https://cloud.ibm.com/apidocs/quantum-computing#error-handling"}],"trace":"6aee1f92-1aee-493d-94bd-e3390edcb4c4"}\n' 
qiskit_runtime_service._create_backend_obj:WARNING:2026-05-05 23:26:39,421: Unable to create configuration for ibm_torino. '404 Client Error: Not Found 

d7h30bq2khts739p7qn0


In [3]:
retrieved_job = service.job(job_id)
retrieved_job.result()

PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=127>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-04-17 13:05:55', stop='2026-04-17 13:05:57', size=4096>)])}, 'version': 2})

# Save results to disk

In [ ]:
import json # Python's built-in JSON library with encoders from Qiskit Runtime.
from qiskit_ibm_runtime import RuntimeEncoder

with open("result.json", "w") as file:
    json.dump(retrieved_job.result(), file, cls=RuntimeEncoder)

In [ ]:
#  load this array from disk in a separate kernel.
from qiskit_ibm_runtime import RuntimeDecoder

with open("result.json", "r") as file:
    result = json.load(file, cls=RuntimeDecoder)

result

PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=127>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-04-17 13:05:55', stop='2026-04-17 13:05:57', size=4096>)])}, 'version': 2})

#### PubResult
It is the result object for a single pub (primitive unified bloc). Each PubResult is a single element of a greater PrimitiveResult. Within this result, there is implementation-defined freeform metadata, and a DataBin in the data field. You typically get instances of this class by iterating over or indexing into a PrimitiveResult, which is what you get from MyPrimitive().run().result().

#### SamplePubResult

This is a SamplerV2-specific subclass of PubResult that adds helper methods to deal with the bit-array like BitArray data directly (implicitly going via the DataBin in data).

#### BasePrimitiveJob

This defines the functionality of the “job handle” object you get by a call to primitive.run(). Typically this object represents a handle to an asynchronous task, where the status(), done(), running(), cancelled() and in_final_state() methods return non-blocking information on the state of the task.

The method result() is typically implemented as a blocking call that waits for the execution result to return. Use of this job object almost invariably ends in a call to result().

#### Subclassing

Each implementer of the primitives should provide a concrete implementation of this interface. There are no provided methods on the base implementation, other than the job_id() getter, since a string key uniquely identifying jobs is a requirement of all primitives.

The ResultT generic type should be set to a subclass of the appropriate versioned primitive result. This typically will mean setting it to PrimitiveResult (for V2), or an implementation-specific subclass of this.

The StatusT generic type is completely freeform; your implementation can provide any status object you like and there is no defined interface. Instead, the done(), running(), cancelled() and in_final_state() methods of this interface should be implemented to give the user simple programmatic access to coarse-grained status information. You can provide additional details in the complete StatusT generic.

Creating this “job” handle is conventionally expected (but not strictly required) to be fast and non-blocking, and for this object to hold an internal asynchronous handle to the actual job. The result() method should typically block until ready, if called before the job has completed.

#### Writing a New Backend

If you have a quantum device or simulator that you would like to integrate with Qiskit you will need to write a backend. A provider is a collection of backends and will provide Qiskit with a method to get available BackendV2 objects. The BackendV2 object provides both information describing a backend and its operation for the transpiler so that circuits can be compiled to something that is optimized and can execute on the backend. It also provides the run() method which can run the QuantumCircuit objects. This enables users and other Qiskit APIs to get results from executing circuits on devices in a standard fashion regardless of how the backend is implemented. At a high level the basic steps for writing a provider are:

Implement a Provider class that handles access to the backend(s).

Implement a BackendV2 subclass and its run() method.

Add any custom gates for the backend’s basis to the session EquivalenceLibrary instance.
Implement a JobV1 subclass that handles interacting with a running job.

#### Provider

A provider class serves a single purpose: to get backend objects that enable executing circuits on a device or simulator. The expectation is that any required credentials and/or authentication will be handled in the initialization of a provider object. The provider object will then provide a list of backends, and methods to filter and acquire backends (using the provided credentials if required). An example provider class looks like:

In [1]:
!pip install qiskit-ibm-runtime --upgrade

In [3]:
from qiskit.providers.providerutils import filter_backends

from backend import MyBackend

class MyProvider:

    def __init__(self, token=None):
        self.token = token
        self.backends = [MyBackend(provider=self)]

    def backends(self, name=None, **kwargs):
        if name:
            filtered_backends = [
                backend for backend in self.backends if backend.name() == name]
            return filter_backends(filtered_backends, **kwargs)
        return filter_backends(self.backends, **kwargs)

#### Backend
The backend classes are the core to the provider. These classes are what provide the interface between Qiskit and the hardware or simulator that will execute circuits. This includes providing the necessary information to describe a backend to the compiler so that it can embed and optimize any circuit for the backend. There are 4 required things in every backend object: a target property to define the model of the backend for the compiler, a max_circuits property to define a limit on the number of circuits the backend can execute in a single batch job (if there is no limit None can be used), a run() method to accept job submissions, and a _default_options method to define the user configurable options and their default values. For example, a minimum working example would be something like:

In [4]:
from qiskit.providers import BackendV2 as Backend
from qiskit.transpiler import Target
from qiskit.providers import Options
from qiskit.circuit import Parameter, Measure
from qiskit.circuit.library import PhaseGate, SXGate, UGate, CXGate, IGate


class Mybackend(Backend):

    def __init__(self):
        super().__init__()

        # Create Target
        self._target = Target("Target for My Backend")
        # Instead of None for this and below instructions you can define
        # a qiskit.transpiler.InstructionProperties object to define properties
        # for an instruction.
        lam = Parameter("λ")
        p_props = {(qubit,): None for qubit in range(5)}
        self._target.add_instruction(PhaseGate(lam), p_props)
        sx_props = {(qubit,): None for qubit in range(5)}
        self._target.add_instruction(SXGate(), sx_props)
        phi = Parameter("φ")
        theta = Parameter("ϴ")
        u_props = {(qubit,): None for qubit in range(5)}
        self._target.add_instruction(UGate(theta, phi, lam), u_props)
        cx_props = {edge: None for edge in [(0, 1), (1, 2), (2, 3), (3, 4)]}
        self._target.add_instruction(CXGate(), cx_props)
        meas_props = {(qubit,): None for qubit in range(5)}
        self._target.add_instruction(Measure(), meas_props)
        id_props = {(qubit,): None for qubit in range(5)}
        self._target.add_instruction(IGate(), id_props)

        # Set option validators
        self.options.set_validator("shots", (1, 4096))
        self.options.set_validator("memory", bool)

    @property
    def target(self):
        return self._target

    @property
    def max_circuits(self):
        return 1024

    @classmethod
    def _default_options(cls):
        return Options(shots=1024, memory=False)

    def run(self, circuits, **kwargs):
        # serialize circuits submit to backend and create a job
        for kwarg in kwargs:
            if not hasattr(self.options, kwarg):
                warnings.warn(
                    "Option %s is not used by this backend" % kwarg,
                    UserWarning, stacklevel=2)
        options = {
            'shots': kwargs.get('shots', self.options.shots),
            'memory': kwargs.get('memory', self.options.memory),
        }
        job_json = convert_to_wire_format(circuit, options)
        job_handle = submit_to_backend(job_json)
        return MyJob(self.job_handle, job_json, circuit)

#### Backend’s Transpiler Interface
The key piece of the Backend object is how it describes itself to the compiler. This is handled with the Target class which defines a model of a backend for the transpiler. A backend object will need to return a Target object from the target attribute which the transpile() function will use as its model of a backend target for compilation.

#### Custom Basis Gates
If your backend doesn’t use gates in the Qiskit circuit library (qiskit.circuit.library) you can integrate support for this into your provider. The basic method for doing this is first to define a Gate subclass for each custom gate in the basis set. For example:

In [5]:
import numpy as np

from qiskit.circuit import Gate
from qiskit.circuit import QuantumCircuit

class SYGate(Gate):
    def __init__(self, label=None):
        super().__init__("sy", 1, [], label=label)

    def _define(self):
        qc = QuantumCircuit(1)
        qc.ry(np.pi / 2, 0)
        self.definition = qc

The key thing to ensure is that for any custom gates in your Backend’s basis set your custom gate’s name attribute (the first param on super().__init__() in the __init__ definition above) does not conflict with the name of any other gates. The name attribute is what is used to identify the gate in the basis set for the transpiler. If there is a conflict the transpiler will not know which gate to use.

Add the custom gate to the target for your backend. This can be done with the Target.add_instruction() method. You’ll need to add an instance of SYGate and its parameters to the target so the transpiler knows it exists. For example, assuming this is part of your BackendV2 implementation for your backend:




In [7]:
from qiskit.transpiler import InstructionProperties

sy_props = {
    (0,): InstructionProperties(duration=2.3e-6, error=0.0002),
    (1,): InstructionProperties(duration=2.1e-6, error=0.0001),
    (2,): InstructionProperties(duration=2.5e-6, error=0.0003),
    (3,): InstructionProperties(duration=2.2e-6, error=0.0004),
}
# self.target.add_instruction(SYGate(), sy_props)  # This is now done in the MyBackend __init__ method

The keys in sy_props define the qubits where the backend SYGate can be used on, and the values define the properties of SYGate on that qubit. For multiqubit gates the tuple keys contain all qubit combinations the gate works on (order is significant, i.e. (0, 1) is different from (1, 0)).

After you’ve defined the custom gates to use for the backend’s basis set then you need to add equivalence rules to the standard equivalence library so that the transpile() function and transpiler module can convert an arbitrary circuit using the custom basis set. This can be done by defining equivalent circuits, in terms of the custom gate, for standard gates. Typically if you can convert from a CXGate (if your basis doesn’t include a standard 2 qubit gate) and some commonly used single qubit rotation gates like the HGate and UGate that should be sufficient for the transpiler to translate any circuit into the custom basis gates. But, the more equivalence rules that are defined from standard gates to your basis the more efficient translation from an arbitrary circuit to the target basis will be (although not always, and there is a diminishing margin of return).

For example, if you were to add some rules for the above custom SYGate we could define the U2Gate and HGate:

In [9]:
import qiskit
from qiskit.circuit.equivalence_library import SessionEquivalenceLibrary
from qiskit.circuit.library import HGate
from qiskit.circuit.library import ZGate
from qiskit.circuit.library import RZGate
from qiskit.circuit.library import U2Gate


# H => Z SY
q = qiskit.QuantumRegister(1, "q")
def_sy_h = qiskit.QuantumCircuit(q)
def_sy_h.append(ZGate(), [q[0]], [])
def_sy_h.append(SYGate(), [q[0]], [])
SessionEquivalenceLibrary.add_equivalence(
    HGate(), def_sy_h)

# u2 => Z SY Z
phi = qiskit.circuit.Parameter('phi')
lam = qiskit.circuit.Parameter('lambda')
q = qiskit.QuantumRegister(1, "q")
def_sy_u2 = qiskit.QuantumCircuit(q)
def_sy_u2.append(RZGate(lam), [q[0]], [])
def_sy_u2.append(SYGate(), [q[0]], [])
def_sy_u2.append(RZGate(phi), [q[0]], [])
SessionEquivalenceLibrary.add_equivalence(
    U2Gate(phi, lam), def_sy_u2)

You will want this to be run on import so that as soon as the provider’s package is imported it will be run. This will ensure that any time the BasisTranslator pass is run with the custom gates the equivalence rules are defined.

It’s also worth noting that depending on the basis you’re using, some optimization passes in the transpiler, such as Optimize1qGatesDecomposition, may not be able to operate with your custom basis. For our SYGate example, the Optimize1qGatesDecomposition will not be able to simplify runs of single qubit gates into the SY basis. This is because the OneQubitEulerDecomposer class does not know how to work in the SY basis. To solve this the SYGate class would need to be added to Qiskit and OneQubitEulerDecomposer updated to support decomposing to the SYGate. Longer term that is likely a better direction for custom basis gates and contributing the definitions and support in the transpiler will ensure that it continues to be well supported by Qiskit moving forward.

#### Custom Transpiler Passes
The transpiler supports the ability for backends to provide custom transpiler stage implementations to facilitate hardware specific optimizations and circuit transformations. Currently there are two stages supported, get_translation_stage_plugin() and get_scheduling_stage_plugin() which allow a backend to specify string plugin names to be used as the default translation and scheduling stages, respectively. These hook points in a BackendV2 class can be used if your backend has requirements for compilation that are not met by the current backend/Target interface. Please also consider submitting a Github issue describing your use case as there is interest in improving these interfaces to be able to describe more hardware architectures in greater depth.

To leverage these hook points you just need to add the methods to your BackendV2 implementation and have them return a string plugin name. For example:

In [11]:
from qiskit.providers import BackendV2

class Mybackend(BackendV2):

    def get_scheduling_stage_plugin(self):
        return "SpecialDD"

    def get_translation_stage_plugin(self):
        return "BasisTranslatorWithCustom1qOptimization"

#### Real-time variables
The transpiler will automatically handle real-time typed classical variables (see qiskit.circuit.classical) and treat the Store instruction as a built-in “directive”, similar to Barrier. No special handling from backends is necessary to permit this.

If your backend is unable to handle classical variables and storage, we recommend that you comment on this in your documentation, and insert a check into your run() method (see Backend.run Method) to eagerly reject circuits containing them. You can examine QuantumCircuit.num_vars for the presence of variables at the top level. If you accept control-flow operations, you might need to recursively search the internal blocks of each for scope-local variables with QuantumCircuit.num_declared_vars.

For example, a function to check for the presence of any manual storage locations, or manual stores to memory:

In [12]:
from qiskit.circuit import Store, ControlFlowOp, QuantumCircuit

def has_realtime_logic(circuit: QuantumCircuit) -> bool:
    if circuit.num_vars:
        return True
    for instruction in circuit.data:
        if isinstance(instruction.operation, Store):
            return True
        elif isinstance(instruction.operation, ControlFlowOp):
            for block in instruction.operation.blocks:
                if has_realtime_logic(block):
                    return True
    return False

#### Angle bounds on Gates
If your backend has constraints on the allowed parameter values for any gate in the target you can model this with angle bounds on the Target. When you add the instruction with the add_instruction() you can use the angle_bounds keyword argument which takes a list of tuples for the upper and lower bound for the parameter of a gate.

For example, this code snippet instead of the example adding the PhaseGate in the example above:

In [14]:
lam = Parameter("λ")
p_props = {(qubit,): None for qubit in range(5)}
#self._target.add_instruction(PhaseGate(lam), p_props, angle_bounds=[(0, math.pi)])

will set the bounds on PhaseGate to be between 0 and π (inclusive). This models the angle constraint in the Target on the angle values for the lam parameter on PhaseGate. The WrapAngles transpiler pass is used to transform any PhaseGate outside the specified angle bounds. You will need to write a function that takes in the angle values for the gate and returns a DAGCircuit. For example:

In [16]:
import math
from qiskit.transpiler.passes import WrapAngles
from typing import List
from qiskit.dagcircuit import DAGCircuit
from qiskit.circuit import Qubit
from qiskit.circuit.library import PhaseGate

def fold_phase(angles: List[float], qubits: List[int]) -> DAGCircuit:
    angle = angles[0]
    if angle > 0:
        number_of_gates = angle / math.pi
    else:
        number_of_gates = (6.28 - angle) / math.pi
    dag = DAGCircuit()
    dag.add_qubits([Qubit()])
    for _ in range(int(number_of_gates)):
        dag.apply_operation_back(PhaseGate(math.pi), [dag.qubits[0]])
    return dag

WrapAngles.DEFAULT_REGISTRY.add_wrapper("phase", fold_phase)

#### Backend.run Method
Of key importance is the run() method, which is used to actually submit circuits to a device or simulator. The run method handles submitting the circuits to the backend to be executed and returning a Job object. Depending on the type of backend this typically involves serializing the circuit object into the API format used by a backend. Since backend serialization needs might differ (and, in the case of local simulators, serialization may not even be needed), it is expected that the backend’s run method will handle this conversion.

An example run method would be something like:

In [17]:
def run(self, circuits, **kwargs):
    for kwarg in kwargs:
        if not hasattr(self.options, kwarg):
            warnings.warn(
                "Option %s is not used by this backend" % kwarg,
                UserWarning, stacklevel=2)
    options = {
        'shots': kwargs.get('shots', self.options.shots),
        'memory': kwargs.get('memory', self.options.memory),
    }
    job_json = convert_to_wire_format(circuit, options)
    job_handle = submit_to_backend(job_json)
    return MyJob(self.job_handle, job_json, circuit)

#### Backend Options
There are often several options for a backend that control how a circuit is run. The typical example of this is something like the number of shots which is how many times the circuit is to be executed. The options available for a backend are defined using an Options object. This object is initially created by the _default_options method of a Backend class. The default options returns an initialized Options object with all the default values for all the options a backend supports. For example, if the backend only supports shots the _default_options method would look like:

In [22]:
@classmethod
def _default_options(cls):
    return Options(shots=1024)

#self.options.set_validator("shots", (1, 4096))

#### Job
The output from the run method is a JobV1 object. Each provider is expected to implement a custom job subclass that defines the behavior for the provider. There are 2 types of jobs depending on the backend’s execution method, either a sync or async. By default jobs are considered async and the expectation is that it represents a handle to the async execution of the circuits submitted with Backend.run(). An async job object provides users the ability to query the status of the execution, cancel a running job, and block until the execution is finished. The result is the primary user facing method which will block until the execution is complete and then will return a Result object with results of the job.

For some backends (mainly local simulators) the execution of circuits is a synchronous operation and there is no need to return a handle to a running job elsewhere. For sync jobs its expected that the run method on the backend will block until a Result object is generated and the sync job will return with that inner Result object.

An example job class for an async API based backend would look something like:




In [19]:
from qiskit.providers import JobV1 as Job
from qiskit.providers import JobError
from qiskit.providers import JobTimeoutError
from qiskit.providers.jobstatus import JobStatus
from qiskit.result import Result


class MyJob(Job):
    def __init__(self, backend, job_id, job_json, circuits):
        super().__init__(backend, job_id)
        self._backend = backend
        self.job_json = job_json
        self.circuits = circuits

    def _wait_for_result(self, timeout=None, wait=5):
        start_time = time.time()
        result = None
        while True:
            elapsed = time.time() - start_time
            if timeout and elapsed >= timeout:
                raise JobTimeoutError('Timed out waiting for result')
            result = get_job_status(self._job_id)
            if result['status'] == 'complete':
                break
            if result['status'] == 'error':
                raise JobError('Job error')
            time.sleep(wait)
        return result

    def result(self, timeout=None, wait=5):
        result = self._wait_for_result(timeout, wait)
        results = [{'success': True, 'shots': len(result['counts']),
                    'data': result['counts']}]
        return Result.from_dict({
            'results': results,
            'backend_name': self._backend.configuration().backend_name,
            'backend_version': self._backend.configuration().backend_version,
            'job_id': self._job_id,
            'success': True,
        })

    def status(self):
        result = get_job_status(self._job_id)
        if result['status'] == 'running':
            status = JobStatus.RUNNING
        elif result['status'] == 'complete':
            status = JobStatus.DONE
        else:
            status = JobStatus.ERROR
        return status

    def submit(self):
        raise NotImplementedError

In [20]:
class MySyncJob(Job):
    _async = False

    def __init__(self, backend, job_id, result):
        super().__init__(backend, job_id)
        self._result = result

    def submit(self):
        return

    def result(self):
        return self._result

    def status(self):
        return JobStatus.DONE

#### Primitives
While not directly part of the provider interface, the qiskit.primitives module is tightly coupled with providers. Specifically the primitive interfaces, such as BaseSampler and BaseEstimator, are designed to enable provider implementations to provide custom implementations which are optimized for the provider’s backends. This can include customizations like circuit transformations, additional pre- and post-processing, batching, caching, error mitigation, etc. The concept of the qiskit.primitives module is to explicitly enable this as the primitive objects are higher level abstractions to produce processed higher level outputs (such as probability distributions and expectation values) that abstract away the mechanics of getting the best result efficiently, to concentrate on higher level applications using these outputs.

For example, if your backends were well suited to leverage mthree measurement mitigation to improve the quality of the results, you could implement a provider-specific Sampler implementation that leverages the M3Mitigation class internally to run the circuits and return quasi-probabilities directly from mthree in the result. Doing this would enable algorithms to get the best results with mitigation applied directly from your backends. You can refer to the documentation in qiskit.primitives on how to write custom implementations. Also, the built-in implementations: Sampler, Estimator, BackendSampler, and BackendEstimator can serve as references/models on how to implement these as well.